In [1]:
#!pip install -U transformers
#!pip install --upgrade gradio
#!pip install "huggingface_hub<1.0.0"
#!pip install huggingface_hub transformers

In [2]:
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import get_json_schema

model_name = "HuggingFaceTB/SmolLM3-3B"
device='cpu' 
# allowed entries Expected one of cpu, cuda (NVIDA), ipu, xpu, mkldnn, opengl(any graphcis card should use), opencl(parallel to cuda runs on anything that is amd (cuda) more generic), ideep, hip, ve, fpga, maia, xla, lazy, vulkan (like opengl - another graphic card language), mps, meta, hpu, mtia, privateuseone device type at start of device string: gpu

#With device='opengl' get error -- RuntimeError: PyTorch is not linked with support for opengl devices
#With device='opencl' get error -- RuntimeError: PyTorch is not linked with support for opencl devices
#With device='vulkan' get error -- RuntimeError: PyTorch is not linked with support for vulkan devices

#This is not a loading with Pytorch issue this is a pytorch can't run those issues according to ai overlord G. Mini at google need to use the ones specfiic to the chip set.

# Mac uses mps HAVE on list -- but keeps restarting the kernel so won't run entire process (my guess is just something something the handling doesn't work in jupiter notebooks). 
# AMD/Intel (Windows) uses Use ROCm (Linux) or DirectML (Windows/WSL).--- which isn't here ugh
# For NVIDIA GPUs: Use CUDA, which is the primary and most optimized backend for PyTorch. HAVE on list
# For Intel GPUs: Use the XPU backend via the Intel Extension for PyTorch (IPEX). HAVE on list

tokenizer = AutoTokenizer.from_pretrained(model_name)


In [3]:
path="/Volumes/data/24.chdi.01-PHASE2/Kathryn_Connectomics/ZQ175_20260923-Female/Scalar_and_Volume/anovan_1/Genotype_AgeofTermination/Non_Erode/Subject_Data_Table.csv"
subject_data = pd.read_csv(path,delimiter='\t',low_memory=False,header=1) 
subject_data = subject_data.drop([0, 1]).reset_index(drop=True)

path="/Volumes/data/24.chdi.01-PHASE2/Kathryn_Connectomics/ZQ175_20260923-Female/Scalar_and_Volume/anovan_1/Genotype_AgeofTermination/Non_Erode/Bilateral_2/Group_Data_Table_Genotype.csv"
group_data = pd.read_csv(path,delimiter='\t',low_memory=False,header=1) 
group_data = group_data.drop([0, 1]).reset_index(drop=True)

path="/Volumes/data/24.chdi.01-PHASE2/Kathryn_Connectomics/ZQ175_20260923-Female/Scalar_and_Volume/anovan_1/Genotype_AgeofTermination/Non_Erode/Bilateral_2/Group_Statistical_Results_Genotype.csv"
group_stats = pd.read_csv(path,delimiter='\t',low_memory=False,header=0) 
group_stats = group_stats.drop([0, 1]).reset_index(drop=True)

#hf_dataset=Dataset.from_pandas(group_stats)
#retriever=RagRetriever.from_pretrained(
#    model_name,
#    index_name="custom",
#    passages_dataset=hf_dataset)

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

In [4]:
def multiply(a: float, b: float):
    """
    A function that multiplies two numbers

    Args:
        a: The first number to multiply
        b: The second number to multiply
    """
    return a * b
def parse_
tools=[multiply]

In [5]:
prompt = "What is 5 times 5."
messages_think = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages_think,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [6]:
generated_ids = model.generate(**model_inputs, max_new_tokens=32768)

In [38]:
import json

output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :]
out=tokenizer.decode(output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

out=out.splitlines()

begin_tool_call=0
for line in out:
    if line == '<tool_call>':
        begin_tool_call=1
    elif line == '</tool_call>':
        begin_tool_call=0
    else:
        my_dict = json.loads(line)
        print(my_dict)
        print(my_dict.get("arguments"))
        
        parse_arguments(my_dict.get("arguments"))
        
        result = globals()[my_dict.get("name")](my_dict.get("arguments"))

{'name': 'multiply', 'arguments': {'a': 5, 'b': 5}}
{'a': 5, 'b': 5}


TypeError: multiply() missing 1 required positional argument: 'b'